### Start Project

In [3]:
# Load Environment Variables
import os
from dotenv import load_dotenv
load_dotenv()
print("Environment variable loaded")

Environment variable loaded


In [4]:
# Imports from Langchain
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from typing import List

In [5]:
## Custom Pre-Processor Function to Process the PDF
class SmartPDFProcessor:
    """Advanced PDF processing with error handling"""
    def __init__(self, chunk_size=1000, chunk_overlap=100):
        self.chunk_size=chunk_size,
        self.chunk_overlap=chunk_overlap,
        self.text_splitter=RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=[" "]
        )

    def process_pdf(self, pdf_path:str)->List[Document]:
        """Process PDF with smart chunking and metadata enhancement"""

        # Load PDF

        loader=PyMuPDFLoader(pdf_path)
        pages=loader.load()

        # Process each page

        processed_chunks=[]

        for page_num, page in enumerate(pages):
            # Clean text
            cleaned_text=self._clean_text(page.page_content)

            # Skip nearly empty pages
            if len(cleaned_text.strip()) < 50:
                continue

            # Create chunks with enhanced metadata
            chunks = self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(cleaned_text)
                }]
            )

            processed_chunks.extend(chunks)

        return processed_chunks

    def _clean_text(self, text: str) -> str:
        """Clean extracted text"""
        # Remove excessive whitespace
        text = " ".join(text.split())

        # Fix common PDF extraction issues
        text = text.replace("fi", "fi")
        text = text.replace("fl", "fl")

        return text

In [6]:
preprocessor=SmartPDFProcessor()
preprocessor

In [7]:
## Process a PDF with Pre-Processor Function and Create Chunks
try:
    smart_chunks=preprocessor.process_pdf("Data/project.pdf")
    print(f"Processed into {len(smart_chunks)} smart chunks")

    # Show enhanced metadata
    if smart_chunks:
        print("\nSample chunk metadata:")
        for key, value in smart_chunks[0].metadata.items():
            print(f" {key}: {value}")

except Exception as e:
    print(f"Processing error: {e}")            

Processed into 28 smart chunks

Sample chunk metadata:
 producer: 
 creator: WPS Docs
 creationdate: 2026-03-20T22:19:24+05:30
 source: Data/project.pdf
 file_path: Data/project.pdf
 total_pages: 14
 format: PDF 1.7
 title: 
 author: Khushi Adhana
 subject: 
 keywords: 
 moddate: 2026-03-20T22:19:24+05:30
 trapped: 
 modDate: D:20260320221924+05'30'
 creationDate: D:20260320221924+05'30'
 page: 1
 chunk_method: smart_pdf_processor
 char_count: 2874


In [8]:
# Print the chunks
print(smart_chunks[0])
print("--------------------------")
print(smart_chunks[5])
print("--------------------------")
print(smart_chunks[11])
print("--------------------------")
print(smart_chunks[26])
print("--------------------------")

page_content=' Abstract The rapid growth of data-driven applications has led to an increasing reliance on relational databases for storing and managing structured information. However, accessing these databases typically requires knowledge of Structured Query Language (SQL), which can be challenging for non-technical users. This thesis proposes a retrieval-augmented Natural Language to SQL system that enables users to interact with relational databases using natural language queries. The proposed system integrates Natural Language Processing (NLP), semantic retrieval, and Large Language Models (LLMs) to automatically translate user questions into executable SQL queries. A FAISS-based vector similarity search mechanism is employed to retrieve relevant Natural Language–SQL examples from a dataset, which are incorporated into the prompt provided to the language model. The SQL generation process is powered by the Gemini 2.5 Flash Large Language Model, which interprets the natural language

### Initialize Huggingface Embedding and Create FAISS Vector Store

In [9]:
# Import required modules
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

In [10]:
# Initialize Huggingface Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
# Create FAISS Vector Store from smart_chunks
vectorstore = FAISS.from_documents(
    documents=smart_chunks,
    embedding=embeddings
)
print(f"\n Embeddings are created and stored in memory")


 Embeddings are created and stored in memory


In [12]:
## Save FAISS Index to Local Disk

FAISS_INDEX_PATH = "faiss_index"

vectorstore.save_local(FAISS_INDEX_PATH)

print("Vector store saved to 'faiss_index' directory")

Vector store saved to 'faiss_index' directory


In [13]:
## Load FAISS Index Later

loaded_vectorstore = FAISS.load_local(
    FAISS_INDEX_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)
print(f"Loaded vector store contains {loaded_vectorstore.index.ntotal} vectors")

Loaded vector store contains 28 vectors


In [14]:
# Quick Similarity Test
query = "What is this document about?"
docs = loaded_vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:300])



--- Result 1 ---
Step 1: Query Input The user enters a natural language question through the Streamlit dashboard. Step 2: Embedding Generation The natural language query is converted into a vector embedding using an embedding model. This embedding represents the semantic meaning of the query. Step 3: Semantic Retrie

--- Result 2 ---
for both technical and non-technical users. Overall, the research highlights the potential of combining retrieval-augmented generation techniques with large language models to develop intelligent natural language interfaces for relational databases. The proposed approach contributes toward making st

--- Result 3 ---
dashboards allows users to query and visualize database information more efficiently. 2.7 Research Gap Despite significant progress in Text-to-SQL research, several challenges remain. Many existing systems either rely solely on language models or require complex fine- tuning on large datasets. Furth


In [15]:
# Load Environment
load_dotenv()

True

In [16]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [34]:
# Define GROQ LLM
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant")

In [35]:
llm.invoke("Explain what is langchain")

AIMessage(content="LangChain is an open-source framework for building large language models (LLMs) into production-ready applications. It provides a set of tools and APIs that make it easier to integrate LLMs into various use cases, such as chatbots, virtual assistants, and other conversational AI systems.\n\nLangChain was created by Jack Visneski and Chris Minnick, and it's primarily developed in the TypeScript programming language. The framework is designed to be modular, allowing developers to use different LLMs, such as those from OpenAI, Google, or Hugging Face, and integrate them into various applications.\n\nKey features of LangChain include:\n\n1. **Modular architecture**: LangChain allows developers to use different LLMs and models, and switch between them as needed.\n2. **Unified API**: LangChain provides a unified API that abstracts away the complexities of working with different LLMs, making it easier to integrate them into applications.\n3. **Task-oriented development**: L

In [40]:
# Define Prompt Template
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context.
Context: {context}

Question: {question}

Answer:""")

In [41]:
# Define Retriever
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [42]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001B0906F9710>, search_kwargs={'k': 3})

### Create RAG Chain using Langchain Expression Language (LCEL)

In [59]:
from typing import List
# Format documents for the prompt
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into prompt"""
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'Unknown')
        formatted.append(f"Document {i+1} (Sorce: {source}):\n {doc.page_content}")
    return "\n\n".join(formatted)    

In [60]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Create RAG Chain

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | simple_prompt
    | llm
    | StrOutputParser()
)

In [52]:
rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001B0906F9710>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context.\nContext: {context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B094BDD550>

In [55]:
response = rag_chain.invoke("Tells about Prompt engineering Strategy?")

print(response)

Unfortunately, the provided context documents do not explicitly mention the term "Prompt engineering Strategy." However, based on the information provided, it appears that the system uses a structured prompt construction in Step 4 of the process described in Document 1, which contains:

1. Database schema
2. Retrieved NL-SQL examples
3. User query

This structured prompt is then passed to the Gemini 2.5 Flash LLM for SQL generation in Step 5. This suggests that the system employs a structured approach to constructing prompts, which could be considered a form of prompt engineering strategy.

In more general terms, the system's approach to prompt construction appears to be focused on combining relevant information from the database schema, retrieved NL-SQL examples, and the user query to inform the generation of the SQL query. This approach may be seen as an example of a prompt engineering strategy that aims to improve the accuracy and usability of the generated SQL queries.


In [57]:
for chunk in rag_chain.stream("Summarize the document"):
    print(chunk, end="", flush=True)

The documents describe a system for developing intelligent natural language interfaces for relational databases. The system combines retrieval-augmented generation techniques with large language models to make structured data access more accessible, efficient, and user-friendly for both technical and non-technical users.

The system consists of several steps: 

1. The user enters a natural language question through a Streamlit dashboard.
2. The query is converted into a vector embedding using an embedding model.
3. The FAISS vector database retrieves the top-k most similar natural language queries from the dataset.
4. A structured prompt is constructed containing the database schema, retrieved NL-SQL examples, the user query, and other information.
5. The prompt is passed to the Gemini 2.5 Flash LLM, which generates the corresponding SQL query.
6. The generated SQL query is executed on the MySQL database.
7. The results are displayed on the Streamlit dashboard in tabular and graphical 

In [58]:
## Query multiple questions (batch)
questions = [
    "What is the document about?",
    "What are key points?",
    "Which problem it solved?"
]

responses = rag_chain.batch(questions)

for q, r, in zip(questions, responses):
    print(f"\nQ: {q}\nA: {r}")


Q: What is the document about?
A: The documents appear to be discussing a research project that aims to develop an intelligent natural language interface for relational databases. The project focuses on combining retrieval-augmented generation techniques with large language models to make structured data access more accessible, efficient, and user-friendly for both technical and non-technical users.

Q: What are key points?
A: Based on the provided context, the key points are:

1. The system combines retrieval-augmented generation techniques with large language models to develop intelligent natural language interfaces for relational databases.
2. The proposed approach makes structured data access more accessible, efficient, and user-friendly for both technical and non-technical users.
3. The system uses a FAISS vector database for semantic retrieval and Gemini 2.5 Flash LLM for SQL generation.
4. The system addresses the limitations of existing Text-to-SQL research, including reliance

# Streamlit Dashboard

In [61]:
pip install streamlit langchain langchain-community langchain-core langchain-groq langchain-huggingface faiss-cpu python-dotenv pymupdf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


# Access to Streamlit Dashboard
### Step 1: Open command prompt
### Step 2: cd folder_path  (Move to folder)
### Step 3: python -m streamlit run app.py 

